# 03 — DB2 on IBM Cloud: Ingestion & Validation
**PIX Fraud Intelligence | IBM Portfolio Project**

**Goals:**
- Connect to DB2 on IBM Cloud Lite via `ibm_db` driver
- Create the `TRANSACTIONS` table schema
- Ingest ~100k processed rows in batches
- Validate with SQL queries and read back into pandas

> **Setup:** Copy `config/credentials_template.json` to `config/credentials.json`  
> and fill in your DB2 on IBM Cloud Lite connection details.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import ibm_db
import ibm_db_dbi
from src.db2_connector import DB2Connector
import warnings
warnings.filterwarnings('ignore')

## 1. Connect to DB2 on IBM Cloud

In [ ]:
conn = DB2Connector(credentials_path='../config/credentials.json')
print('Connected to DB2 on IBM Cloud!')

# Sanity check
result = conn.query('SELECT CURRENT TIMESTAMP AS ts FROM SYSIBM.SYSDUMMY1')
print(f'DB2 server time: {result["TS"].iloc[0]}')

## 2. Load Processed Data

In [ ]:
df = pd.read_csv('../data/processed/transactions_features.csv')
print(f'Shape: {df.shape} | Fraud rate: {df["Class"].mean():.4%}')
df.head(3)

## 3. Create Table Schema
We define a typed schema based on the DataFrame columns.

In [ ]:
# Drop table if exists (idempotent)
try:
    conn.execute('DROP TABLE TRANSACTIONS')
    print('Existing table dropped.')
except Exception:
    print('Table did not exist, creating fresh.')

# Build CREATE TABLE DDL dynamically from DataFrame dtypes
def pandas_dtype_to_db2(dtype):
    if dtype == 'int64' or dtype == 'int32':
        return 'INTEGER'
    elif dtype == 'float64' or dtype == 'float32':
        return 'DOUBLE'
    elif dtype == 'bool':
        return 'SMALLINT'
    else:
        return 'VARCHAR(32)'

col_defs = []
for col in df.columns:
    col_clean = col.replace('-', '_').upper()
    col_type = pandas_dtype_to_db2(str(df[col].dtype))
    col_defs.append(f'  {col_clean} {col_type}')

ddl = 'CREATE TABLE TRANSACTIONS (\n' + ',\n'.join(col_defs) + '\n)'
print(ddl)

conn.execute(ddl)
print('\nTable TRANSACTIONS created.')

## 4. Batch Insert

In [ ]:
# Rename columns to match DB2 DDL (uppercase, no hyphens)
df_db2 = df.rename(columns=lambda c: c.replace('-', '_').upper())

print(f'Inserting {len(df_db2):,} rows into TRANSACTIONS...')
conn.insert_dataframe(df_db2, table='TRANSACTIONS', batch_size=2000)
print('\nIngestion complete!')

## 5. Validation — SQL Queries

In [ ]:
# Row count
count_df = conn.query('SELECT COUNT(*) AS TOTAL FROM TRANSACTIONS')
print(f'Total rows in DB2: {count_df["TOTAL"].iloc[0]:,}')

# Fraud breakdown
fraud_df = conn.query('''
    SELECT CLASS, COUNT(*) AS CNT,
           CAST(COUNT(*) AS DOUBLE) / SUM(COUNT(*)) OVER () * 100 AS PCT
    FROM TRANSACTIONS
    GROUP BY CLASS
    ORDER BY CLASS
''')
print('\nClass distribution in DB2:')
print(fraud_df)

# Split breakdown
split_df = conn.query('''
    SELECT SPLIT, CLASS, COUNT(*) AS CNT
    FROM TRANSACTIONS
    GROUP BY SPLIT, CLASS
    ORDER BY SPLIT, CLASS
''')
print('\nTrain/test split:')
print(split_df)

In [ ]:
# Read training set back into pandas (for local baseline)
train_from_db2 = conn.query('SELECT * FROM TRANSACTIONS WHERE SPLIT = \'train\'')
print(f'Training set from DB2: {train_from_db2.shape}')
train_from_db2.head(3)

## 6. Close Connection

In [ ]:
conn.close()
print('DB2 connection closed.')